# Notebook 24 — WU-20E independent review & release gate

**CPU only.** Reads WU20B/C/D existing private evidence, five full-fold calibration files, 15 PNGs and the human-review CSV. Generates a candidate-policy digest and fail-closed release status. **No training, labels, data staging, model inference or clinical approval.**

**Important:** The reviewer CSV currently contains **15 PENDING decisions**. The notebook will report **BLOCKED** until human review and other prerequisites are met, by design. No code can substitute for the independent reviewer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
B=ROOT/'results'/'wu20b_full_fold_calibration'
C=ROOT/'results'/'wu20c_nnunet_runtime_qualification'
D=ROOT/'results'/'wu20d_full_native_volume_qualification'
OUTPUT=ROOT/'results'/'wu20e_review_and_release_gate'
SPLITS=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'splits_final.json'
ARGS=['--wu20b',str(B/'WU20B_aggregate.json'),
      '--wu20c',str(C/'WU20C_probe_aggregate.json'),
      '--wu20d',str(D/'WU20D_four_case_aggregate_PRIVATE.json'),
      '--folds',str(B/'fold_private'), '--splits',str(SPLITS),
      '--review',str(C/'WU20C_mask_review_checklist_PRIVATE.csv'),
      '--overlays',str(B/'private_mask_overlays'), '--output',str(OUTPUT)]
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
assert sha(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
for relative in (B/'WU20B_aggregate.json', C/'WU20C_probe_aggregate.json', D/'WU20D_four_case_aggregate_PRIVATE.json'):
    assert relative.is_file(),relative
assert len(list((B/'private_mask_overlays').glob('*_support_QC.png')))==15
print('Frozen split, audits and overlay inventory: FOUND')

## Run the signed-evidence *preflight* (never approves training)

The script source is pinned to the audited GitHub commit. No raw patient images are uploaded to GitHub.

In [ ]:
COMMIT='dee292ab855e1f37a8258f0875f89d6e61fb78aa'
SOURCE=Path('/content/wu20e_review_gate.py')
URL=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}/scripts/audit_isles2024_wu20e_release_gate.py'
subprocess.run(['curl','-fsSL','--retry','5','--retry-delay','2',URL,'-o',str(SOURCE)],check=True)
assert SOURCE.stat().st_size>5000
OUTPUT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,str(SOURCE),*ARGS],check=True)
report=json.loads((OUTPUT/'WU20E_release_gate_aggregate.json').read_text())
assert report['training_ready'] is False
print('Release state:',report['status'])
print('Reviewer decisions:',report['review_decisions'])
print('Policy digest:',report['candidate_policy_sha256'])

## How to continue

1. Provide the **private WU-20E anatomical atlas** and original native images to a qualified independent reviewer. The three slices per case are a screening aid, not proof of full-volume anatomical correctness.
2. Reviewer fills the pre-existing **private** CSV with evidence and notes; do not overwrite it with an autogenerated blank file.
3. Obtain separately signed method/policy review for the mask and q0.5/q99.5, train-only median and ±8 cap. Verify professional credentials outside this automated tool.
4. Re-run this notebook to update the **blocked/pending evidence** state. Human approval and later WU-20F technical staging are still required before any training release.

No automatic clinical sign-off is performed or implied.

In [ ]:
private=OUTPUT/'WU20E_release_gate_PRIVATE.json'
status=json.loads(private.read_text())
print('All gating entries:')
for reason in status['blocked_release_gates']:
    print(' -',reason)
print('PRIVATE report:',private)
print('NO training authorization: ',status['training_ready'])